# Hindi Text Processing and Machine Translation

## Objective

The project aims to process seven files containing shuffled and unpunctuated Hindi text.

The main objectives are:
1. Restore punctuation in the Hindi text.
2. Segment the text into individual sentences.
3. Identify semantically related sentences and infer their logical order.
4. Group ordered sentences into batches of up to five.
5. Translate the Hindi sentences into English using pretrained neural machine translation models.
6. Evaluate translation quality and reconstruct the final translated batches.

## Approach

The pipeline uses pretrained models for punctuation restoration, multilingual sentence embeddings, and machine translation. No models are trained from scratch.

**Environment:** Google Colab  
**Execution:** CPU-only

In [1]:
import os
import re
import gc
import zipfile
import warnings
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

# CPU-only configuration
os.environ["CUDA_VISIBLE_DEVICES"] = ""

DEVICE = "cpu"
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("GPU available:", torch.cuda.is_available())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nEnvironment initialized successfully.")

ENVIRONMENT CHECK
PyTorch: 2.14.1+cpu
Device: cpu
GPU available: False
NumPy: 2.4.6
Pandas: 2.2.3

Environment initialized successfully.


In [2]:
from google.colab import files

print("Upload the original Hindi dataset ZIP:")
uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]

assert len(zip_files) == 1, "Please upload exactly one ZIP file."

ZIP_PATH = Path("/content") / zip_files[0]

print("\nUploaded successfully:", ZIP_PATH.name)

Upload the original Hindi dataset ZIP:


Saving Sample-Raw-No-Punctuations.zip to Sample-Raw-No-Punctuations (3).zip

Uploaded successfully: Sample-Raw-No-Punctuations (3).zip


In [3]:
BASE_DIR = Path("/content/hindi_text_project")
DATA_DIR = BASE_DIR / "dataset"
OUTPUT_DIR = BASE_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

sample_files = sorted(DATA_DIR.rglob("sample-*.txt"))

assert len(sample_files) == 7, f"Expected 7 files, found {len(sample_files)}"

print("Dataset extracted successfully.")
print("Number of sample files:", len(sample_files))

for file in sample_files:
    print("✓", file.name)

Dataset extracted successfully.
Number of sample files: 7
✓ sample-1.txt
✓ sample-2.txt
✓ sample-3.txt
✓ sample-4.txt
✓ sample-5.txt
✓ sample-6.txt
✓ sample-7.txt


In [4]:
dataset_records = []
original_texts = {}

for file in sample_files:
    text = file.read_text(encoding="utf-8")
    original_texts[file.stem] = text

    dataset_records.append({
        "Sample": file.stem,
        "Characters": len(text),
        "Words": len(text.split()),
        "NULL markers": text.count("NULL")
    })

dataset_df = pd.DataFrame(dataset_records)

print("=" * 65)
print("ORIGINAL DATASET STATISTICS")
print("=" * 65)

display(dataset_df)

print("\nTotal samples:", len(dataset_df))
print("Total characters:", dataset_df["Characters"].sum())
print("Total words:", dataset_df["Words"].sum())
print("Total NULL markers:", dataset_df["NULL markers"].sum())

print("\nSample Hindi text:")
print(original_texts["sample-1"][:600])

ORIGINAL DATASET STATISTICS


,Sample,Characters,Words,NULL markers
0,sample-1,3797,752,5
1,sample-2,3825,773,5
2,sample-3,3931,748,6
3,sample-4,5428,1122,4
4,sample-5,4992,1035,9
5,sample-6,2330,518,11
6,sample-7,3913,873,27



Total samples: 7
Total characters: 28216
Total words: 5821
Total NULL markers: 67

Sample Hindi text:
भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का नाम कांचीपुरम है यमुना तट पर भगवान श्रीकृष्ण की अवतार भ


In [5]:
!pip install -q transformers sentencepiece accelerate

In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

PUNCT_MODEL = "Horizon-Labs/punctuation-restoration-small"

punct_tokenizer = AutoTokenizer.from_pretrained(PUNCT_MODEL)
punct_model = AutoModelForTokenClassification.from_pretrained(PUNCT_MODEL)

punct_model.to(DEVICE)
punct_model.eval()

print("Punctuation restoration model loaded successfully.")
print("Model:", PUNCT_MODEL)
print("Device:", DEVICE)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Punctuation restoration model loaded successfully.
Model: Horizon-Labs/punctuation-restoration-small
Device: cpu


In [7]:


# Download the model's official punctuation helper
from huggingface_hub import hf_hub_download
import importlib.util

helper_path = hf_hub_download(
    "Horizon-Labs/punctuation-restoration-small",
    "punctuate.py"
)

spec = importlib.util.spec_from_file_location(
    "punctuate",
    helper_path
)

punctuate_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(punctuate_module)

# Initialize the punctuation restoration model
punct_model = punctuate_module.Punctuator(
    "Horizon-Labs/punctuation-restoration-small",
    device="cpu"
)

print("Punctuation restoration model initialized.")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Punctuation restoration model initialized.


In [8]:
punctuated_texts = {}

for sample, original_text in original_texts.items():

    restored_text = punct_model(original_text)

    punctuated_texts[sample] = restored_text

    print(
        f"{sample}: "
        f"{len(original_text)} -> {len(restored_text)} characters"
    )

print("\nAll seven samples processed.")

sample-1: 3797 -> 3850 characters
sample-2: 3825 -> 3882 characters
sample-3: 3931 -> 3989 characters
sample-4: 5428 -> 5505 characters
sample-5: 4992 -> 5078 characters
sample-6: 2330 -> 2422 characters
sample-7: 3913 -> 3999 characters

All seven samples processed.


In [9]:
PUNCT_DIR = OUTPUT_DIR / "punctuated"
PUNCT_DIR.mkdir(parents=True, exist_ok=True)

for sample, text in punctuated_texts.items():

    output_path = PUNCT_DIR / f"{sample}_punctuated.txt"

    output_path.write_text(text, encoding="utf-8")

print("Punctuated files saved:", len(punctuated_texts))

Punctuated files saved: 7


In [10]:
segmented_data = {}

for sample, text in punctuated_texts.items():

    parts = re.split(r"(?<=[।?!])\s+|(NULL)", text)

    segments = [
        part.strip()
        for part in parts
        if part and part.strip()
    ]

    segmented_data[sample] = segments

    sentence_count = sum(s != "NULL" for s in segments)
    null_count = segments.count("NULL")

    print(
        f"{sample}: {sentence_count} sentences | "
        f"{null_count} NULL markers"
    )

print("\nSegmentation completed.")

sample-1: 53 sentences | 5 NULL markers
sample-2: 53 sentences | 5 NULL markers
sample-3: 51 sentences | 6 NULL markers
sample-4: 54 sentences | 4 NULL markers
sample-5: 58 sentences | 9 NULL markers
sample-6: 49 sentences | 11 NULL markers
sample-7: 65 sentences | 27 NULL markers

Segmentation completed.


In [11]:
sentence_records = []

for sample, segments in segmented_data.items():

    for position, segment in enumerate(segments, start=1):

        sentence_records.append({
            "sample": sample,
            "segment_id": position,
            "text": segment,
            "status": "NULL" if segment == "NULL" else "TEXT"
        })

sentences_df = pd.DataFrame(sentence_records)

print("Total segments:", len(sentences_df))
print("Text segments:", (sentences_df["status"] == "TEXT").sum())
print("NULL markers:", (sentences_df["status"] == "NULL").sum())

display(sentences_df.head(10))

Total segments: 450
Text segments: 383
NULL markers: 67


,sample,segment_id,text,status
0,sample-1,1,भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्र...,TEXT
1,sample-1,2,चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कां...,TEXT
2,sample-1,3,द्वारका चार धामों में एक धाम भी है।,TEXT
3,sample-1,4,कहते हैं कि उज्जैन में महाकाल को नमस्कार कर ले...,TEXT
4,sample-1,5,गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभ...,TEXT
5,sample-1,6,अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि...,TEXT
6,sample-1,7,स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का ना...,TEXT
7,sample-1,8,यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का य...,TEXT
8,sample-1,9,दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जान...,TEXT
9,sample-1,10,ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के ...,TEXT


In [12]:
sentences_df.to_csv(
    OUTPUT_DIR / "segmented_hindi_sentences.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Segmented dataset saved successfully.")

Segmented dataset saved successfully.


In [13]:
!pip install -q sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(
    EMBED_MODEL,
    device="cpu"
)

print("Sentence embedding model loaded.")
print("Model:", EMBED_MODEL)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence embedding model loaded.
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embedding dimension: 384


In [15]:
# Exclude NULL markers; retain their records in sentences_df
text_df = sentences_df[sentences_df["status"] == "TEXT"].copy()

sentence_texts = text_df["text"].tolist()

embeddings = embedding_model.encode(
    sentence_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

text_df["embedding_index"] = range(len(text_df))

print("\nEmbedding generation completed.")
print("Number of sentences:", len(sentence_texts))
print("Embedding matrix shape:", embeddings.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]


Embedding generation completed.
Number of sentences: 383
Embedding matrix shape: (383, 384)


In [16]:
assert embeddings.shape[0] == len(text_df)
assert embeddings.shape[1] == 384
assert np.isfinite(embeddings).all()

np.save(OUTPUT_DIR / "sentence_embeddings.npy", embeddings)

text_df.to_csv(
    OUTPUT_DIR / "sentence_embedding_index.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Embedding validation successful.")
print("Embeddings saved.")

Embedding validation successful.
Embeddings saved.


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_results = {}

for sample in text_df["sample"].unique():

    sample_indices = text_df[
        text_df["sample"] == sample
    ]["embedding_index"].to_numpy()

    sample_embeddings = embeddings[sample_indices]

    similarity_matrix = cosine_similarity(sample_embeddings)

    similarity_results[sample] = {
        "indices": sample_indices,
        "matrix": similarity_matrix
    }

print("Semantic similarity matrices generated.")

for sample, result in similarity_results.items():
    print(sample, ":", result["matrix"].shape)

Semantic similarity matrices generated.
sample-1 : (53, 53)
sample-2 : (53, 53)
sample-3 : (51, 51)
sample-4 : (54, 54)
sample-5 : (58, 58)
sample-6 : (49, 49)
sample-7 : (65, 65)


In [18]:
similarity_candidates = []

for sample, result in similarity_results.items():

    indices = result["indices"]
    matrix = result["matrix"]

    for i, global_i in enumerate(indices):

        similarities = matrix[i].copy()
        similarities[i] = -1

        top_positions = np.argsort(similarities)[-3:][::-1]

        for rank, j in enumerate(top_positions, start=1):

            global_j = indices[j]

            similarity_candidates.append({
                "sample": sample,
                "sentence_id": int(text_df.iloc[global_i]["segment_id"]),
                "related_sentence_id": int(text_df.iloc[global_j]["segment_id"]),
                "similarity": float(similarities[j]),
                "rank": rank
            })

similarity_df = pd.DataFrame(similarity_candidates)

print("Similarity candidate pairs:", len(similarity_df))
display(similarity_df.head(15))

Similarity candidate pairs: 1149


,sample,sentence_id,related_sentence_id,similarity,rank
0,sample-1,1,24,0.745266,1
1,sample-1,1,39,0.743620,2
2,sample-1,1,25,0.739330,3
3,sample-1,2,18,0.885130,1
4,sample-1,2,17,0.784983,2
5,sample-1,2,19,0.769528,3
6,sample-1,3,50,0.665279,1
7,sample-1,3,8,0.622200,2
8,sample-1,3,39,0.607289,3
9,sample-1,4,11,0.399372,1


In [19]:
strongest_pairs = (
    similarity_df[similarity_df["sentence_id"] < similarity_df["related_sentence_id"]]
    .sort_values("similarity", ascending=False)
)

for _, row in strongest_pairs.head(20).iterrows():

    sample = row["sample"]
    id1 = int(row["sentence_id"])
    id2 = int(row["related_sentence_id"])

    sentence1 = text_df[
        (text_df["sample"] == sample) &
        (text_df["segment_id"] == id1)
    ]["text"].iloc[0]

    sentence2 = text_df[
        (text_df["sample"] == sample) &
        (text_df["segment_id"] == id2)
    ]["text"].iloc[0]

    print("=" * 80)
    print(f"{sample} | Similarity: {row['similarity']:.4f}")
    print(f"Sentence {id1}: {sentence1}")
    print(f"Sentence {id2}: {sentence2}")

sample-5 | Similarity: 0.9288
Sentence 8: यह सुनहले रंग का रेशम मल्टीवोल्टाइन वर्ग के रेशम कीड़ों से प्राप्‍त होता है।
Sentence 34: यह सफेद या लाल रंग का रेशम है जो मल्टीवोल्टाइन वर्ग के रेशम कीड़ों से प्राप्‍त होता है।
sample-1 | Similarity: 0.9186
Sentence 24: ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
Sentence 39: गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।
sample-5 | Similarity: 0.9070
Sentence 8: यह सुनहले रंग का रेशम मल्टीवोल्टाइन वर्ग के रेशम कीड़ों से प्राप्‍त होता है।
Sentence 48: ऐसे रेशम कीट जिनसे टसर रेशम प्राप्‍त होता है
sample-5 | Similarity: 0.8965
Sentence 21: कश्मीर में यूनिवोल्टाइन तथा बाइवोल्टाइन रेशम कीट
Sentence 24: , पश्‍चिम बंगाल में बढ़िया किस्म के मल्टीवोल्टाइन रेशम कीट
sample-3 | Similarity: 0.8888
Sentence 46: भारत में कुल 12 चक्रवर्ती सम्राट हुए।
Sentence 48: भारत के कुल 12 चक्रवर्ती सम्राट में से 6 का जन्म स्थान हस्तिनापुर है।
sample-5 | Similarity: 0.8860
Sentence 27: तथा कर्नाटक में मल्टीवोल्टाइन रेशम कीट का पालन होता है।
Sentence 4

In [20]:
from sklearn.cluster import AgglomerativeClustering

clustered_records = []

for sample, result in similarity_results.items():

    indices = result["indices"]
    sample_embeddings = embeddings[indices]

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=0.35
    )

    cluster_labels = clustering.fit_predict(sample_embeddings)

    for global_index, cluster_id in zip(indices, cluster_labels):

        row = text_df.iloc[global_index]

        clustered_records.append({
            "sample": sample,
            "sentence_id": int(row["segment_id"]),
            "cluster_id": int(cluster_id),
            "hindi": row["text"]
        })

clustered_df = pd.DataFrame(clustered_records)

print("Candidate clustering completed.")
print("Total sentences:", len(clustered_df))
print("Total candidate groups:", clustered_df.groupby(
    ["sample", "cluster_id"]
).ngroups)

print("\nGroups per sample:")
display(
    clustered_df.groupby("sample")["cluster_id"]
    .nunique()
    .reset_index(name="candidate_groups")
)

Candidate clustering completed.
Total sentences: 383
Total candidate groups: 284

Groups per sample:


,sample,candidate_groups
0,sample-1,35
1,sample-2,33
2,sample-3,34
3,sample-4,40
4,sample-5,42
5,sample-6,41
6,sample-7,59


In [21]:
for sample in sorted(clustered_df["sample"].unique()):

    print("\n" + "=" * 85)
    print(sample.upper())
    print("=" * 85)

    sample_data = clustered_df[clustered_df["sample"] == sample]

    for cluster_id, group in sample_data.groupby("cluster_id"):

        print(f"\nGROUP {cluster_id} | {len(group)} sentences")

        for _, row in group.sort_values("sentence_id").iterrows():
            print(f"ID {row['sentence_id']}: {row['hindi']}")


SAMPLE-1

GROUP 0 | 2 sentences
ID 20: गंगा नदी पर तैरती दीपमालाओं का दृश्य देखते ही बनता है।
ID 56: शिप्रा अथवा क्षिप्रा नदी के तट पर बसी महाकाल की यह उज्जैन नगरी पतित पावनी है।

GROUP 1 | 5 sentences
ID 1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
ID 22: उस समय गढ़ मुक्तेश्वर बल्लभ संप्रदाय का मुख्य केन्द्र था।
ID 24: ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
ID 25: महाराज शिव ने भगवान परशुराम से यहाँ शिव मंदिर की स्थापना कराई थी।
ID 39: गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।

GROUP 2 | 2 sentences
ID 27: उत्तर रेलवे की मुगलसराय लखनऊ लाइन के मुगलसराय स्टेशन से 128 मील की दूरी पर अयोध्या स्टेशन है।
ID 31: मध्य रेलवे की मुंबई भोपाल दिल्ली लाइन के भोपाल स्टेशन से एक लाइन उज्जैन जाती है।

GROUP 3 | 2 sentences
ID 6: अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है।
ID 29: अयोध्या पवित्र नदी सरयू के तट पर स्थित है।

GROUP 4 | 3 sentences
ID 11: ऐसा कहा गया है कि चतुर्मास में इन सप्तपुरियों का दर्शन मोक्ष प्रदा

In [22]:
print("=" * 65)
print("CURRENT NOTEBOOK VARIABLES")
print("=" * 65)

keywords = ["embed", "sentence", "cluster", "group", "sample"]

for name, obj in globals().copy().items():
    if any(word in name.lower() for word in keywords):
        if not name.startswith("_"):
            try:
                print(f"{name}: {type(obj).__name__}, shape={obj.shape}")
            except:
                print(f"{name}: {type(obj).__name__}")

CURRENT NOTEBOOK VARIABLES
sample_files: list
sample: str
sentence_count: int
sentence_records: list
sentences_df: DataFrame, shape=(450, 4)
SentenceTransformer: ABCMeta
EMBED_MODEL: str
embedding_model: SentenceTransformer
sentence_texts: list
embeddings: ndarray, shape=(383, 384)
sample_indices: ndarray, shape=(65,)
sample_embeddings: ndarray, shape=(65, 384)
sentence1: str
sentence2: str
AgglomerativeClustering: type
clustered_records: list
clustering: AgglomerativeClustering
cluster_labels: ndarray, shape=(65,)
cluster_id: int
clustered_df: DataFrame, shape=(383, 4)
sample_data: DataFrame, shape=(65, 4)
group: DataFrame, shape=(1, 4)


In [23]:
# ============================================================
# SIMILARITY SCORE ANALYSIS
# ============================================================

pair_records = []

for sample, result in similarity_results.items():

    indices = result["indices"]
    matrix = result["matrix"]

    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):

            global_i = indices[i]
            global_j = indices[j]

            row1 = text_df.iloc[global_i]
            row2 = text_df.iloc[global_j]

            pair_records.append({
                "sample": sample,
                "sentence_id_1": int(row1["segment_id"]),
                "sentence_id_2": int(row2["segment_id"]),
                "similarity": float(matrix[i, j]),
                "sentence_1": row1["text"],
                "sentence_2": row2["text"]
            })

pair_similarity_df = pd.DataFrame(pair_records)

print("=" * 65)
print("SIMILARITY SCORE DISTRIBUTION")
print("=" * 65)

print(
    pair_similarity_df["similarity"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(4)
)

print("\n" + "=" * 65)
print("TOP 25 MOST SIMILAR SENTENCE PAIRS")
print("=" * 65)

top_pairs = pair_similarity_df.nlargest(25, "similarity")

for _, row in top_pairs.iterrows():

    print(f"\n{row['sample']} | "
          f"IDs {row['sentence_id_1']} and {row['sentence_id_2']}")

    print(f"Similarity: {row['similarity']:.4f}")
    print(f"1. {row['sentence_1']}")
    print(f"2. {row['sentence_2']}")
    print("-" * 65)

SIMILARITY SCORE DISTRIBUTION
count    10371.0000
mean         0.2620
std          0.1802
min         -0.1494
25%          0.1285
50%          0.2353
75%          0.3772
90%          0.5167
95%          0.6016
99%          0.7367
max          0.9288
Name: similarity, dtype: float64

TOP 25 MOST SIMILAR SENTENCE PAIRS

sample-5 | IDs 8 and 34
Similarity: 0.9288
1. यह सुनहले रंग का रेशम मल्टीवोल्टाइन वर्ग के रेशम कीड़ों से प्राप्‍त होता है।
2. यह सफेद या लाल रंग का रेशम है जो मल्टीवोल्टाइन वर्ग के रेशम कीड़ों से प्राप्‍त होता है।
-----------------------------------------------------------------

sample-1 | IDs 24 and 39
Similarity: 0.9186
1. ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
2. गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।
-----------------------------------------------------------------

sample-5 | IDs 8 and 48
Similarity: 0.9070
1. यह सुनहले रंग का रेशम मल्टीवोल्टाइन वर्ग के रेशम कीड़ों से प्राप्‍त होता है।
2. ऐसे रेशम कीट जिनसे टसर रेशम प्राप्‍त होता है


In [24]:
# ============================================================
# HIGH-CONFIDENCE RECIPROCAL SENTENCE LINKS
# ============================================================

SIMILARITY_THRESHOLD = 0.60
TOP_K = 5

# Rank neighbours separately for each sentence
ranked_pairs = pair_similarity_df.copy()

ranked_pairs["rank_from_1"] = (
    ranked_pairs.groupby(["sample", "sentence_id_1"])["similarity"]
    .rank(method="first", ascending=False)
)

ranked_pairs["rank_from_2"] = (
    ranked_pairs.groupby(["sample", "sentence_id_2"])["similarity"]
    .rank(method="first", ascending=False)
)

# Keep only reciprocal top-K pairs above threshold
candidate_links_df = ranked_pairs[
    (ranked_pairs["similarity"] >= SIMILARITY_THRESHOLD) &
    (ranked_pairs["rank_from_1"] <= TOP_K) &
    (ranked_pairs["rank_from_2"] <= TOP_K)
].copy()

candidate_links_df = candidate_links_df.sort_values(
    ["sample", "similarity"],
    ascending=[True, False]
).reset_index(drop=True)

print("=" * 65)
print("RECIPROCAL HIGH-CONFIDENCE CANDIDATE LINKS")
print("=" * 65)

print("Similarity threshold:", SIMILARITY_THRESHOLD)
print("Mutual nearest-neighbour limit:", TOP_K)
print("Candidate links:", len(candidate_links_df))

print("\nLinks per sample:")
print(candidate_links_df.groupby("sample").size())

print("\n" + "=" * 65)
print("CANDIDATE LINKS FOR REVIEW")
print("=" * 65)

for _, row in candidate_links_df.iterrows():

    print(f"\n{row['sample']} | "
          f"IDs {int(row['sentence_id_1'])} and "
          f"{int(row['sentence_id_2'])}")

    print(f"Similarity: {row['similarity']:.4f}")
    print(f"1. {row['sentence_1']}")
    print(f"2. {row['sentence_2']}")
    print("-" * 65)

RECIPROCAL HIGH-CONFIDENCE CANDIDATE LINKS
Similarity threshold: 0.6
Mutual nearest-neighbour limit: 5
Candidate links: 356

Links per sample:
sample
sample-1    79
sample-2    65
sample-3    56
sample-4    59
sample-5    53
sample-6    28
sample-7    16
dtype: int64

CANDIDATE LINKS FOR REVIEW

sample-1 | IDs 24 and 39
Similarity: 0.9186
1. ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
2. गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।
-----------------------------------------------------------------

sample-1 | IDs 2 and 18
Similarity: 0.8851
1. चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
2. चेंगलपट स्टेशन से एक लाइन अरकोनम तक जाती है।
-----------------------------------------------------------------

sample-1 | IDs 22 and 39
Similarity: 0.8665
1. उस समय गढ़ मुक्तेश्वर बल्लभ संप्रदाय का मुख्य केन्द्र था।
2. गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।
-----------------------------------------------------------------

sample-1 | IDs 28 and 58
Similarit

In [25]:
# Conservative semantic matching:
# A pair must be in each other's top 3 matches
# and have cosine similarity >= 0.60.

candidate_edges = similarity_df.copy()

candidate_edges["id_1"] = candidate_edges[
    ["sentence_id", "related_sentence_id"]
].min(axis=1)

candidate_edges["id_2"] = candidate_edges[
    ["sentence_id", "related_sentence_id"]
].max(axis=1)

mutual_pairs = (
    candidate_edges
    .groupby(["sample", "id_1", "id_2"])
    .agg(
        appearances=("similarity", "count"),
        similarity=("similarity", "mean")
    )
    .reset_index()
)

candidate_pairs_df = mutual_pairs[
    (mutual_pairs["appearances"] == 2) &
    (mutual_pairs["similarity"] >= 0.60)
].copy()

# Attach the original Hindi sentences
sentence_lookup = text_df.set_index(
    ["sample", "segment_id"]
)["text"].to_dict()

candidate_pairs_df["sentence_1"] = candidate_pairs_df.apply(
    lambda r: sentence_lookup[(r["sample"], r["id_1"])], axis=1
)

candidate_pairs_df["sentence_2"] = candidate_pairs_df.apply(
    lambda r: sentence_lookup[(r["sample"], r["id_2"])], axis=1
)

candidate_pairs_df = candidate_pairs_df.sort_values(
    ["sample", "similarity"],
    ascending=[True, False]
).reset_index(drop=True)

print("HIGH-CONFIDENCE CANDIDATE PAIRS")
print("=" * 80)
print("Total pairs:", len(candidate_pairs_df))

print("\nPairs per sample:")
print(candidate_pairs_df.groupby("sample").size())

for _, row in candidate_pairs_df.iterrows():
    print("\n" + "-" * 80)
    print(f'{row["sample"]} | IDs {row["id_1"]} and {row["id_2"]}')
    print(f'Similarity: {row["similarity"]:.4f}')
    print("1.", row["sentence_1"])
    print("2.", row["sentence_2"])

HIGH-CONFIDENCE CANDIDATE PAIRS
Total pairs: 171

Pairs per sample:
sample
sample-1    26
sample-2    31
sample-3    24
sample-4    31
sample-5    23
sample-6    20
sample-7    16
dtype: int64

--------------------------------------------------------------------------------
sample-1 | IDs 24 and 39
Similarity: 0.9186
1. ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
2. गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।

--------------------------------------------------------------------------------
sample-1 | IDs 2 and 18
Similarity: 0.8851
1. चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
2. चेंगलपट स्टेशन से एक लाइन अरकोनम तक जाती है।

--------------------------------------------------------------------------------
sample-1 | IDs 22 and 39
Similarity: 0.8665
1. उस समय गढ़ मुक्तेश्वर बल्लभ संप्रदाय का मुख्य केन्द्र था।
2. गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।

--------------------------------------------------------------------------------
sample-1 | 

In [26]:
from collections import defaultdict

candidate_groups = {}

for sample in text_df["sample"].unique():

    sample_sentences = text_df[
        text_df["sample"] == sample
    ].copy()

    parent = {
        int(sid): int(sid)
        for sid in sample_sentences["segment_id"]
    }

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        root_a = find(a)
        root_b = find(b)

        if root_a != root_b:
            parent[root_b] = root_a

    sample_pairs = candidate_pairs_df[
        candidate_pairs_df["sample"] == sample
    ]

    for _, pair in sample_pairs.iterrows():
        union(int(pair["id_1"]), int(pair["id_2"]))

    groups = defaultdict(list)

    for sid in parent:
        groups[find(sid)].append(sid)

    candidate_groups[sample] = sorted(
        [sorted(ids) for ids in groups.values()],
        key=lambda ids: ids[0]
    )

print("=" * 70)
print("CONSERVATIVE CANDIDATE GROUPING")
print("=" * 70)

total_groups = 0

for sample, groups in candidate_groups.items():
    total_groups += len(groups)
    multi_sentence = sum(len(g) > 1 for g in groups)
    standalone = sum(len(g) == 1 for g in groups)

    print(
        f"{sample}: {len(groups)} groups | "
        f"{multi_sentence} multi-sentence | "
        f"{standalone} standalone"
    )

print("\nTotal candidate groups:", total_groups)
print(
    "Total sentences accounted for:",
    sum(len(g) for groups in candidate_groups.values() for g in groups)
)

CONSERVATIVE CANDIDATE GROUPING
sample-1: 33 groups | 7 multi-sentence | 26 standalone
sample-2: 30 groups | 7 multi-sentence | 23 standalone
sample-3: 30 groups | 8 multi-sentence | 22 standalone
sample-4: 27 groups | 6 multi-sentence | 21 standalone
sample-5: 41 groups | 7 multi-sentence | 34 standalone
sample-6: 33 groups | 7 multi-sentence | 26 standalone
sample-7: 50 groups | 8 multi-sentence | 42 standalone

Total candidate groups: 244
Total sentences accounted for: 383


In [27]:
# Evaluate coherence of every multi-sentence candidate group

sentence_embedding_lookup = {
    (row["sample"], int(row["segment_id"])):
        embeddings[int(row["embedding_index"])]
    for _, row in text_df.iterrows()
}

sentence_text_lookup = {
    (row["sample"], int(row["segment_id"])): row["text"]
    for _, row in text_df.iterrows()
}

coherence_records = []

for sample, groups in candidate_groups.items():

    for group_number, sentence_ids in enumerate(groups, start=1):

        if len(sentence_ids) < 2:
            continue

        group_embeddings = np.array([
            sentence_embedding_lookup[(sample, sid)]
            for sid in sentence_ids
        ])

        pairwise_matrix = cosine_similarity(group_embeddings)

        upper_triangle = pairwise_matrix[
            np.triu_indices(len(sentence_ids), k=1)
        ]

        coherence_records.append({
            "sample": sample,
            "group_id": group_number,
            "sentence_ids": sentence_ids,
            "size": len(sentence_ids),
            "minimum_similarity": float(upper_triangle.min()),
            "mean_similarity": float(upper_triangle.mean())
        })

coherence_df = pd.DataFrame(coherence_records)

print("=" * 75)
print("CANDIDATE GROUP COHERENCE SUMMARY")
print("=" * 75)

print("Multi-sentence groups:", len(coherence_df))
print("\nMinimum pairwise similarity:")
print(coherence_df["minimum_similarity"].describe())

print("\nMean pairwise similarity:")
print(coherence_df["mean_similarity"].describe())

print("\nGroups with weakest internal coherence:")
display(
    coherence_df.sort_values("minimum_similarity")
    [["sample", "group_id", "size",
      "minimum_similarity", "mean_similarity"]]
    .head(10)
)

print("\n" + "=" * 75)
print("INSPECTING THE 10 WEAKEST GROUPS")
print("=" * 75)

for _, row in coherence_df.nsmallest(
    10, "minimum_similarity"
).iterrows():

    print(
        f'\n{row["sample"]} | Group {row["group_id"]} '
        f'| IDs {row["sentence_ids"]}'
    )
    print(
        f'Minimum similarity: {row["minimum_similarity"]:.3f} '
        f'| Mean similarity: {row["mean_similarity"]:.3f}'
    )

    for sid in row["sentence_ids"]:
        print(f'  ID {sid}: {sentence_text_lookup[(row["sample"], sid)]}')

CANDIDATE GROUP COHERENCE SUMMARY
Multi-sentence groups: 50

Minimum pairwise similarity:
count    50.000000
mean      0.588891
std       0.180629
min      -0.042630
25%       0.547615
50%       0.641700
75%       0.683084
max       0.888818
Name: minimum_similarity, dtype: float64

Mean pairwise similarity:
count    50.000000
mean      0.669700
std       0.094199
min       0.420543
25%       0.615356
50%       0.673686
75%       0.740896
max       0.888818
Name: mean_similarity, dtype: float64

Groups with weakest internal coherence:


,sample,group_id,size,minimum_similarity,mean_similarity
23,sample-4,2,11,-0.042630,0.420543
40,sample-6,13,3,0.212175,0.480930
26,sample-4,7,13,0.222735,0.545635
10,sample-2,13,10,0.232827,0.548187
35,sample-6,1,6,0.286689,0.513789
19,sample-3,16,5,0.300286,0.547851
47,sample-7,16,7,0.379046,0.534858
15,sample-3,2,8,0.395081,0.571647
38,sample-6,9,3,0.398133,0.549377
3,sample-1,8,7,0.428592,0.655566



INSPECTING THE 10 WEAKEST GROUPS

sample-4 | Group 2 | IDs [2, 3, 15, 26, 27, 34, 35, 41, 43, 46, 57]
Minimum similarity: -0.043 | Mean similarity: 0.421
  ID 2: रेशम कई प्रकार के कीड़ों द्वारा बनाया जाता है, परंतु इन सब में बांबिक्स मोरी Bombyx Mori जाति का रेशम कीड़ा ही प्रमुख है और सेरीकल्चर फार्म के विशेष वातारण में रेशम बनाता है।
  ID 3: अब विश्‍व में रेशम उत्पादन करने वाले देशों में भारत का प्रमुख स्थान है।
  ID 15: रेशम का कीड़ा देखने में तितली की तरह होता है, जिसके अंडों से पहले लारवा कैटरपिलर बनता है।
  ID 26: इस प्रकार रेशम के कीड़े का जीवन चक्र पूरा हो जाता है।
  ID 27: इन कीड़ों की मादा मॉथ moth कोये से निकलने के तीन दिन बाद करीब 700 अंडे देती है और फिर मर जाती है।
  ID 34: भारत ही एक ऐसा देश है जहाँ कई प्रकार के रेशम के कीड़ों का पालन करके विविध प्रकार का रेशम प्राप्‍त किया जाता है।
  ID 35: राज्यों में प्रति एकड़ उत्पादन के हिसाब से महाराष्‍ट्र सबसे अग्रणी है किंतु कुल उत्पादन में गुजरात सूची में सर्वोपरि है।
  ID 41: उन्होंने बताया कि हम बांबिक्स मोरी जाति के ही रेशम कीड़ों का

In [28]:
# Complete-link refinement with a maximum group size of 5

MAX_GROUP_SIZE = 5
MIN_INTERNAL_SIMILARITY = 0.50

refined_groups = {}
refinement_records = []

for sample in text_df["sample"].unique():

    sample_data = text_df[
        text_df["sample"] == sample
    ].sort_values("segment_id")

    sentence_ids = sample_data["segment_id"].astype(int).tolist()

    sample_embeddings = np.array([
        sentence_embedding_lookup[(sample, sid)]
        for sid in sentence_ids
    ])

    sim_matrix = cosine_similarity(sample_embeddings)

    id_to_position = {
        sid: pos for pos, sid in enumerate(sentence_ids)
    }

    # Begin with every sentence as its own group
    groups = {sid: {sid} for sid in sentence_ids}

    # Map each sentence to its current group representative
    owner = {sid: sid for sid in sentence_ids}

    sample_edges = candidate_pairs_df[
        candidate_pairs_df["sample"] == sample
    ].sort_values("similarity", ascending=False)

    for _, edge in sample_edges.iterrows():

        a = int(edge["id_1"])
        b = int(edge["id_2"])

        root_a = owner[a]
        root_b = owner[b]

        if root_a == root_b:
            continue

        group_a = groups[root_a]
        group_b = groups[root_b]

        # Do not exceed the required group size
        if len(group_a) + len(group_b) > MAX_GROUP_SIZE:
            continue

        # Check every cross-group sentence pair
        cross_similarities = [
            sim_matrix[id_to_position[x], id_to_position[y]]
            for x in group_a
            for y in group_b
        ]

        if min(cross_similarities) < MIN_INTERNAL_SIMILARITY:
            continue

        # Merge the two groups
        merged = group_a | group_b

        del groups[root_a]
        del groups[root_b]

        new_root = min(merged)
        groups[new_root] = merged

        for sid in merged:
            owner[sid] = new_root

    refined_groups[sample] = sorted(
        [sorted(group) for group in groups.values()],
        key=lambda group: group[0]
    )

    for group_number, group in enumerate(
        refined_groups[sample], start=1
    ):
        refinement_records.append({
            "sample": sample,
            "group_id": group_number,
            "sentence_ids": group,
            "size": len(group)
        })

refined_groups_df = pd.DataFrame(refinement_records)

print("=" * 70)
print("REFINED CANDIDATE GROUPING")
print("=" * 70)

for sample, groups in refined_groups.items():
    multi = sum(len(g) > 1 for g in groups)
    standalone = sum(len(g) == 1 for g in groups)

    print(
        f"{sample}: {len(groups)} groups | "
        f"{multi} multi-sentence | {standalone} standalone"
    )

print("\nTotal groups:", len(refined_groups_df))
print(
    "Multi-sentence groups:",
    (refined_groups_df["size"] > 1).sum()
)
print(
    "Standalone sentences:",
    (refined_groups_df["size"] == 1).sum()
)
print(
    "Largest group:",
    refined_groups_df["size"].max()
)
print(
    "Total sentences accounted for:",
    refined_groups_df["size"].sum()
)

REFINED CANDIDATE GROUPING
sample-1: 36 groups | 8 multi-sentence | 28 standalone
sample-2: 33 groups | 8 multi-sentence | 25 standalone
sample-3: 34 groups | 9 multi-sentence | 25 standalone
sample-4: 34 groups | 11 multi-sentence | 23 standalone
sample-5: 42 groups | 8 multi-sentence | 34 standalone
sample-6: 38 groups | 8 multi-sentence | 30 standalone
sample-7: 53 groups | 10 multi-sentence | 43 standalone

Total groups: 270
Multi-sentence groups: 62
Standalone sentences: 208
Largest group: 5
Total sentences accounted for: 383


In [29]:
print("=" * 90)
print("REFINED MULTI-SENTENCE CANDIDATE GROUPS")
print("=" * 90)

for sample, groups in refined_groups.items():

    print(f"\n{'=' * 90}")
    print(sample.upper())
    print("=" * 90)

    group_number = 0

    for group in groups:

        if len(group) < 2:
            continue

        group_number += 1

        print(f"\nGROUP {group_number} | Sentence IDs: {group}")

        for sid in group:
            print(f"  ID {sid}: {sentence_text_lookup[(sample, sid)]}")

        print("-" * 75)

REFINED MULTI-SENTENCE CANDIDATE GROUPS

SAMPLE-1

GROUP 1 | Sentence IDs: [1, 22, 24, 25, 39]
  ID 1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
  ID 22: उस समय गढ़ मुक्तेश्वर बल्लभ संप्रदाय का मुख्य केन्द्र था।
  ID 24: ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
  ID 25: महाराज शिव ने भगवान परशुराम से यहाँ शिव मंदिर की स्थापना कराई थी।
  ID 39: गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।
---------------------------------------------------------------------------

GROUP 2 | Sentence IDs: [2, 17, 18, 28, 58]
  ID 2: चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
  ID 17: उक्त लाइन के महू स्टेशन से भी एक लाइन उज्जैन को गई है।
  ID 18: चेंगलपट स्टेशन से एक लाइन अरकोनम तक जाती है।
  ID 28: पूर्वोत्तर रेलवे की आगरा फोर्ट से गोरखपुर जाने वाली लाइन तथा पश्चिम रेलवे की मुंबई कोटा दिल्ली लाइन पर मथुरा स्टेशन है चलें गढ़ मुक्तेश्वर का गंगा मेला लक्सर से एक लाइन हरिद्वार तक गई है।
  ID 58: पश्चिम रेलवे की मुंब

In [30]:
# Audit potentially incomplete or incorrectly segmented sentences

segmentation_issues = []

for _, row in sentences_df.iterrows():

    text = str(row["text"]).strip()

    if row["status"] != "TEXT":
        continue

    issues = []

    # Very short fragments
    if len(text.split()) <= 3:
        issues.append("Very short fragment")

    # No terminal punctuation
    if not re.search(r"[।?!]$", text):
        issues.append("No ending punctuation")

    # Starts with punctuation
    if re.match(r"^[,،:;]", text):
        issues.append("Starts with punctuation")

    # Possible multiple sentences in one segment
    if len(re.findall(r"[।?!]", text)) > 1:
        issues.append("Multiple sentence endings")

    # Suspiciously long segment
    if len(text.split()) > 45:
        issues.append("Unusually long segment")

    if issues:
        segmentation_issues.append({
            "sample": row["sample"],
            "segment_id": row["segment_id"],
            "word_count": len(text.split()),
            "issues": "; ".join(issues),
            "text": text
        })

segmentation_audit_df = pd.DataFrame(segmentation_issues)

print("=" * 85)
print("SENTENCE SEGMENTATION AUDIT")
print("=" * 85)

print("Total text segments:", len(text_df))
print("Potentially problematic segments:", len(segmentation_audit_df))

print("\nIssues by type:")
for issue in [
    "Very short fragment",
    "No ending punctuation",
    "Starts with punctuation",
    "Multiple sentence endings",
    "Unusually long segment"
]:
    count = segmentation_audit_df["issues"].str.contains(
        issue, regex=False
    ).sum()
    print(f"{issue}: {count}")

print("\nPotentially problematic segments:")
for _, row in segmentation_audit_df.iterrows():
    print(
        f'\n{row["sample"]} | ID {row["segment_id"]} '
        f'| {row["issues"]}'
    )
    print(row["text"])

SENTENCE SEGMENTATION AUDIT
Total text segments: 383
Potentially problematic segments: 59

Issues by type:
Very short fragment: 10
No ending punctuation: 30
Starts with punctuation: 24
Multiple sentence endings: 0
Unusually long segment: 0

Potentially problematic segments:

sample-1 | ID 13 | No ending punctuation
पूर्वोत्तर रेलवे की एक लाइन भटनी से

sample-1 | ID 33 | No ending punctuation
खांडवी वन में भगवान परशुराम द्वारा स्थापित शिव बल्लभपुर नामक स्थान पर मुक्तेश्वर महादेव की पूजा

sample-1 | ID 35 | Very short fragment; No ending punctuation
यज्ञ

sample-1 | ID 47 | No ending punctuation
मुगलसराय से अमृतसर तथा देहरादून जाने वाली मुख्य रेलवे लाइन पर मुगलसराय स्टेशन से 7 मील पर काशी

sample-1 | ID 51 | Very short fragment
गढ़।

sample-2 | ID 13 | No ending punctuation
अगले दिन प्रात पूर्णमासी को सभी ने गंगा स्नान कर पूजा अर्चना

sample-2 | ID 15 | Very short fragment
कथा आदि की।

sample-2 | ID 16 | No ending punctuation
प्रत्येक वर्ष लाखों की संख्या में श्रद्धालु गढ़ मुक्तेश्वर कार्

In [31]:
# Inspect context around all flagged segments without modifying the data

print("=" * 95)
print("SEGMENTATION CONTEXT REVIEW")
print("=" * 95)

for _, issue in segmentation_audit_df.iterrows():

    sample = issue["sample"]
    sid = int(issue["segment_id"])

    sample_segments = sentences_df[
        sentences_df["sample"] == sample
    ].sort_values("segment_id").reset_index(drop=True)

    current_pos = sample_segments.index[
        sample_segments["segment_id"] == sid
    ].tolist()

    if not current_pos:
        continue

    pos = current_pos[0]

    print("\n" + "-" * 95)
    print(f'{sample} | ID {sid} | {issue["issues"]}')

    if pos > 0:
        prev_row = sample_segments.iloc[pos - 1]
        print(
            f'PREVIOUS [{prev_row["segment_id"]}]: '
            f'{prev_row["text"]}'
        )

    print(f'CURRENT  [{sid}]: {issue["text"]}')

    if pos < len(sample_segments) - 1:
        next_row = sample_segments.iloc[pos + 1]
        print(
            f'NEXT     [{next_row["segment_id"]}]: '
            f'{next_row["text"]}'
        )

SEGMENTATION CONTEXT REVIEW

-----------------------------------------------------------------------------------------------
sample-1 | ID 13 | No ending punctuation
PREVIOUS [12]: हिंदू धर्म में तीर्थ का बड़ा महत्व है।
CURRENT  [13]: पूर्वोत्तर रेलवे की एक लाइन भटनी से
NEXT     [14]: NULL

-----------------------------------------------------------------------------------------------
sample-1 | ID 33 | No ending punctuation
PREVIOUS [32]: गाजियाबाद जिले के गढ़ मुक्तेश्वर में पतित पावनी गंगा के तट पर हर साल कार्तिक पूर्णिमा के अवसर पर लगने वाले उत्तर भारत के प्रसिद्ध और प्राचीन धार्मिक मेले का इतिहास लगभग पाँच हजार वर्ष पुराना है।
CURRENT  [33]: खांडवी वन में भगवान परशुराम द्वारा स्थापित शिव बल्लभपुर नामक स्थान पर मुक्तेश्वर महादेव की पूजा
NEXT     [34]: NULL

-----------------------------------------------------------------------------------------------
sample-1 | ID 35 | Very short fragment; No ending punctuation
PREVIOUS [34]: NULL
CURRENT  [35]: यज्ञ
NEXT     [36]: NULL

----------

In [32]:
# Classify segmentation audit cases without modifying source sentences

def classify_issue(row):

    sample = row["sample"]
    sid = int(row["segment_id"])
    text = str(row["text"]).strip()

    if text.startswith(":"):
        return "Dialogue formatting"

    if len(text.split()) <= 3:
        return "Short fragment - preserve"

    if not re.search(r"[।?!]$", text):
        return "Missing ending punctuation - review"

    if text.startswith((",", "،", ";")):
        return "Leading punctuation - review"

    return "Other - review"


segmentation_audit_df["review_category"] = (
    segmentation_audit_df.apply(classify_issue, axis=1)
)

print("=" * 75)
print("SEGMENTATION REVIEW CATEGORIES")
print("=" * 75)

print(
    segmentation_audit_df["review_category"]
    .value_counts()
)

print("\nCATEGORY COUNTS BY SAMPLE")

display(
    pd.crosstab(
        segmentation_audit_df["sample"],
        segmentation_audit_df["review_category"]
    )
)

print("\nSENTENCES REQUIRING MANUAL ATTENTION")

display(
    segmentation_audit_df[
        segmentation_audit_df["review_category"].isin([
            "Missing ending punctuation - review",
            "Leading punctuation - review"
        ])
    ][["sample", "segment_id", "review_category", "text"]]
)

SEGMENTATION REVIEW CATEGORIES
review_category
Missing ending punctuation - review    27
Dialogue formatting                    22
Short fragment - preserve              10
Name: count, dtype: int64

CATEGORY COUNTS BY SAMPLE


review_category,Dialogue formatting,Missing ending punctuation - review,Short fragment - preserve
sample,,,
sample-1,0,3,2
sample-2,0,4,1
sample-3,0,3,2
sample-4,0,4,0
sample-5,0,6,0
sample-6,8,1,4
sample-7,14,6,1



SENTENCES REQUIRING MANUAL ATTENTION


,sample,segment_id,review_category,text
0,sample-1,13,Missing ending punctuation - review,पूर्वोत्तर रेलवे की एक लाइन भटनी से
1,sample-1,33,Missing ending punctuation - review,खांडवी वन में भगवान परशुराम द्वारा स्थापित शिव...
3,sample-1,47,Missing ending punctuation - review,मुगलसराय से अमृतसर तथा देहरादून जाने वाली मुख्...
5,sample-2,13,Missing ending punctuation - review,अगले दिन प्रात पूर्णमासी को सभी ने गंगा स्नान ...
7,sample-2,16,Missing ending punctuation - review,प्रत्येक वर्ष लाखों की संख्या में श्रद्धालु गढ़...
8,sample-2,29,Missing ending punctuation - review,गढ़ मुक्तेश्वर में धार्मिक महत्व की कई जगहें ह...
9,sample-2,46,Missing ending punctuation - review,इस पर शंकर जी कहा कि हे देवी सतयुग में सभी तीर...
10,sample-3,18,Missing ending punctuation - review,कार्तिक पूर्णिमा और वैशाख शुक्ल तृतीया को श्वे...
12,sample-3,39,Missing ending punctuation - review,श्वेतांबर ट्रस्ट धर्मशाला में ठहरने के लिए 400...
13,sample-3,52,Missing ending punctuation - review,अतिशय क्षेत्र तिजारा हस्तिनापुर को शास्त्रों म...


In [33]:
# Finalize segmentation review
segmentation_audit_df.to_csv(
    OUTPUT_DIR / "segmentation_review.csv",
    index=False,
    encoding="utf-8-sig"
)

sentences_df.to_csv(
    OUTPUT_DIR / "final_segmented_sentences.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Segmentation finalized.")
print("Text segments:", (sentences_df["status"] == "TEXT").sum())
print("NULL markers:", (sentences_df["status"] == "NULL").sum())
print("Review cases:", len(segmentation_audit_df))
print("Files saved successfully.")

Segmentation finalized.
Text segments: 383
NULL markers: 67
Review cases: 59
Files saved successfully.


In [34]:
# Consolidate refined semantic groups into a structured table

group_records = []

for sample, groups in refined_groups.items():
    for group_number, sentence_ids in enumerate(groups, start=1):

        group_records.append({
            "sample": sample,
            "group_id": f"{sample}_G{group_number:02d}",
            "sentence_ids": sentence_ids,
            "group_size": len(sentence_ids),
            "hindi_text": "\n".join(
                sentence_lookup[(sample, sid)]
                for sid in sentence_ids
            )
        })

candidate_groups_df = pd.DataFrame(group_records)

candidate_groups_df.to_csv(
    OUTPUT_DIR / "candidate_groups.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Candidate groups created:", len(candidate_groups_df))
print("Multi-sentence groups:", (candidate_groups_df["group_size"] > 1).sum())
print("Standalone sentences:", (candidate_groups_df["group_size"] == 1).sum())
print("Largest group:", candidate_groups_df["group_size"].max())
print("All text sentences accounted for:",
      sum(candidate_groups_df["group_size"]) == len(text_df))

Candidate groups created: 270
Multi-sentence groups: 62
Standalone sentences: 208
Largest group: 5
All text sentences accounted for: True


In [36]:
# Compatibility fix for IndicTransToolkit and newer Transformers

import transformers.tokenization_utils as tokenization_utils
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

tokenization_utils.PreTrainedTokenizerBase = PreTrainedTokenizerBase

from IndicTransToolkit import IndicProcessor
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("IndicTransToolkit imported successfully.")

IndicTransToolkit imported successfully.


In [38]:
# Load public Hindi-to-English translation model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-hi-en"

tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL
)

translation_model.eval()
DEVICE = "cpu"
translation_model.to(DEVICE)

print("Translation model loaded successfully!")
print("Model:", TRANSLATION_MODEL)
print("Device:", DEVICE)

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  304MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  304MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Translation model loaded successfully!
Model: Helsinki-NLP/opus-mt-hi-en
Device: cpu


In [39]:
# Translate all Hindi text segments into English

translated_texts = []

BATCH_SIZE = 8

for start in range(0, len(sentence_texts), BATCH_SIZE):

    batch = sentence_texts[start:start + BATCH_SIZE]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = translation_model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=2
        )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    translated_texts.extend(translations)

    if (start // BATCH_SIZE + 1) % 10 == 0:
        print(f"Translated {len(translated_texts)}/{len(sentence_texts)}")

assert len(translated_texts) == len(text_df)

translation_df = text_df[
    ["sample", "segment_id", "text"]
].copy()

translation_df["english_translation"] = translated_texts

translation_df.to_csv(
    OUTPUT_DIR / "sentence_translations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTranslation completed!")
print("Total translated:", len(translation_df))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Translated 80/383


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Translated 160/383


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Translated 240/383


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Translated 320/383


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Translation completed!
Total translated: 383


In [40]:
# Create bilingual grouped output

translation_lookup = translation_df.set_index(
    ["sample", "segment_id"]
)["english_translation"].to_dict()

final_group_records = []

for _, group in candidate_groups_df.iterrows():

    sample = group["sample"]
    sentence_ids = group["sentence_ids"]

    english_parts = [
        translation_lookup[(sample, sid)]
        for sid in sentence_ids
    ]

    final_group_records.append({
        "sample": sample,
        "group_id": group["group_id"],
        "sentence_ids": sentence_ids,
        "group_size": len(sentence_ids),
        "hindi_text": group["hindi_text"],
        "english_translation": " ".join(english_parts)
    })

final_groups_df = pd.DataFrame(final_group_records)

final_groups_df.to_csv(
    OUTPUT_DIR / "final_bilingual_groups.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Bilingual groups created:", len(final_groups_df))
print("Translated sentences:", len(translation_df))
print("Groups containing multiple sentences:",
      (final_groups_df["group_size"] > 1).sum())
print("Standalone groups:",
      (final_groups_df["group_size"] == 1).sum())

Bilingual groups created: 270
Translated sentences: 383
Groups containing multiple sentences: 62
Standalone groups: 208


In [41]:
# Inspect representative Hindi-English translations

for sample in sorted(translation_df["sample"].unique()):

    sample_data = translation_df[
        translation_df["sample"] == sample
    ].head(3)

    print("\n" + "=" * 85)
    print(sample.upper())
    print("=" * 85)

    for _, row in sample_data.iterrows():
        print(f'\nID {row["segment_id"]}')
        print("HINDI:", row["text"])
        print("ENGLISH:", row["english_translation"])


SAMPLE-1

ID 1
HINDI: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
ENGLISH: God's ancestors Sir Shiva lived in his van-Security at the Free World.

ID 2
HINDI: चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
ENGLISH: From the Cheungeoff station, the residence is the Zonham station on the line to Arconum.

ID 3
HINDI: द्वारका चार धामों में एक धाम भी है।
ENGLISH: The door is also a dwelling in four dwellings.

SAMPLE-2

ID 1
HINDI: प्रमुख स्थानों पर क्लोज सर्किट टीवी कैमरे भी लगाए गए हैं।
ENGLISH: The Clarge circuit TV cameras on the main places have also been introduced.

ID 2
HINDI: गंगा तट के रेतीले मैदान में मेला लगता है किंतु गढ़ मुक्तेश्वर के गंगा मंदिर में अब 86 सीढ़ियाँ ही बची हैं।
ENGLISH: In the sand plain of the river coast, there are now 86 steps left at the river of Freer's temple.

ID 3
HINDI: मेला स्थल दिल्ली से लगभग 90 किलोमीटर पड़ता है।
ENGLISH: The site takes about 90 miles from Delhi.

SAMPLE-3

ID 1
HINDI: कश्

In [42]:
import gc

del translation_model
del tokenizer
gc.collect()

221

In [43]:
TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL,
    src_lang="hin_Deva",
    tgt_lang="eng_Latn"
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL,
    low_cpu_mem_usage=True
)

translation_model.eval()
translation_model.to("cpu")

TARGET_TOKEN_ID = tokenizer.convert_tokens_to_ids("eng_Latn")

print("NLLB loaded successfully!")
print("Source: Hindi")
print("Target: English")
print("Device: CPU")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB loaded successfully!
Source: Hindi
Target: English
Device: CPU


In [44]:
test_df = (
    text_df.sort_values(["sample", "segment_id"])
    .groupby("sample", as_index=False)
    .head(1)
    .copy()
)

test_texts = test_df["text"].tolist()

inputs = tokenizer(
    test_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512
)

with torch.no_grad():
    outputs = translation_model.generate(
        **inputs,
        forced_bos_token_id=TARGET_TOKEN_ID,
        max_new_tokens=128,
        num_beams=4
    )

test_translations = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)

for (_, row), translation in zip(test_df.iterrows(), test_translations):
    print(f"\n{row['sample']} | Sentence ID: {row['segment_id']}")
    print("HINDI:", row["text"])
    print("ENGLISH:", translation)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



sample-1 | Sentence ID: 1
HINDI: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
ENGLISH: Maharaja Shiva, the ancestor of Lord Rama, spent his time at Vanipurasth Fort Mukteswar.

sample-2 | Sentence ID: 1
HINDI: प्रमुख स्थानों पर क्लोज सर्किट टीवी कैमरे भी लगाए गए हैं।
ENGLISH: Closed-circuit TV cameras have also been installed in key locations.

sample-3 | Sentence ID: 1
HINDI: कश्मीरी गेट से लगातार मेरठ तक की बसें जाती हैं।
ENGLISH: Buses pass from Kashmir Gate to Meerut continuously.

sample-4 | Sentence ID: 1
HINDI: वैसे अन्य जाति के रेशम कीट 350 400 अंडे देते हैं।
ENGLISH: Other silkworms, on the other hand, lay 350,400 eggs.

sample-5 | Sentence ID: 1
HINDI: एक कैटरपिलर अपने वजन से 7 000 - 9 000 गुना शहतूत के पत्‍ते खा जाता है।
ENGLISH: A caterpillar eats 7 000 - 9 000 times its own weight.

sample-6 | Sentence ID: 1
HINDI: यू नो।
ENGLISH: You know.

sample-7 | Sentence ID: 1
HINDI: जी, आप मुझे बता सकती हैं कि प्रोफ़ेसर चौधरी किस कमरे में 

In [45]:
# Replace the earlier OPUS-MT translations with NLLB translations

translated_texts = []
BATCH_SIZE = 4

for start in range(0, len(sentence_texts), BATCH_SIZE):
    batch = sentence_texts[start:start + BATCH_SIZE]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = translation_model.generate(
            **inputs,
            forced_bos_token_id=TARGET_TOKEN_ID,
            max_length=256,
            num_beams=2
        )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    translated_texts.extend(translations)

    if (start // BATCH_SIZE + 1) % 10 == 0:
        print(f"Translated {len(translated_texts)}/{len(sentence_texts)}")

assert len(translated_texts) == len(sentence_texts)

translation_df = text_df[
    ["sample", "segment_id", "text"]
].copy()

translation_df["english_translation"] = translated_texts

translation_df.to_csv(
    OUTPUT_DIR / "sentence_translations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTranslation completed!")
print("Total sentences translated:", len(translation_df))

Translated 40/383
Translated 80/383
Translated 120/383
Translated 160/383
Translated 200/383
Translated 240/383
Translated 280/383
Translated 320/383
Translated 360/383

Translation completed!
Total sentences translated: 383


In [46]:
translation_lookup = translation_df.set_index(
    ["sample", "segment_id"]
)["english_translation"].to_dict()

final_group_records = []

for _, group in candidate_groups_df.iterrows():
    sample = group["sample"]
    sentence_ids = group["sentence_ids"]

    english_parts = [
        translation_lookup[(sample, sid)]
        for sid in sentence_ids
    ]

    final_group_records.append({
        "sample": sample,
        "group_id": group["group_id"],
        "sentence_ids": sentence_ids,
        "group_size": len(sentence_ids),
        "hindi_text": group["hindi_text"],
        "english_translation": " ".join(english_parts)
    })

final_groups_df = pd.DataFrame(final_group_records)

final_groups_df.to_csv(
    OUTPUT_DIR / "final_bilingual_groups.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final bilingual groups updated!")
print("Total groups:", len(final_groups_df))
print("Translated sentences:", len(translation_df))
print("Multi-sentence groups:", (final_groups_df["group_size"] > 1).sum())
print("Standalone groups:", (final_groups_df["group_size"] == 1).sum())

Final bilingual groups updated!
Total groups: 270
Translated sentences: 383
Multi-sentence groups: 62
Standalone groups: 208


In [47]:
 import zipfile
from pathlib import Path

# Verify final results
assert len(translation_df) == 383
assert len(final_groups_df) == 270
assert len(sentences_df[sentences_df["status"] == "NULL"]) == 67

print("FINAL PROJECT SUMMARY")
print("=" * 45)
print("Samples processed:", len(original_texts))
print("Text sentences:", len(translation_df))
print("NULL markers preserved:", 67)
print("Candidate groups:", len(final_groups_df))
print("Multi-sentence groups:", (final_groups_df["group_size"] > 1).sum())
print("Standalone groups:", (final_groups_df["group_size"] == 1).sum())

# Package all generated outputs
ZIP_OUTPUT = BASE_DIR / "Hindi_Text_Processing_Outputs.zip"

with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(BASE_DIR))

print("\nSubmission package created:", ZIP_OUTPUT)
print("Package size:", round(ZIP_OUTPUT.stat().st_size / (1024 * 1024), 2), "MB")

FINAL PROJECT SUMMARY
Samples processed: 7
Text sentences: 383
NULL markers preserved: 67
Candidate groups: 270
Multi-sentence groups: 62
Standalone groups: 208

Submission package created: /content/hindi_text_project/Hindi_Text_Processing_Outputs.zip
Package size: 0.67 MB


In [48]:
from google.colab import files
files.download(str(ZIP_OUTPUT))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
import re
import pandas as pd

def translation_issues(row):
    hindi = str(row["text"]).strip()
    english = str(row["english_translation"]).strip()
    issues = []

    if not english:
        issues.append("Empty translation")

    words = english.split()
    if len(words) >= 10:
        unique_ratio = len(set(w.lower() for w in words)) / len(words)
        if unique_ratio < 0.30:
            issues.append("Excessive repetition")

    if len(words) >= 15:
        for i in range(len(words) - 9):
            chunk = [w.lower() for w in words[i:i+10]]
            if len(set(chunk)) <= 2:
                issues.append("Repeated phrase")
                break

    if len(words) <= 2 and len(hindi.split()) >= 8:
        issues.append("Suspiciously short translation")

    return ", ".join(issues)

translation_review_df = translation_df.copy()
translation_review_df["review_issue"] = translation_review_df.apply(
    translation_issues, axis=1
)

translation_review_df = translation_review_df[
    translation_review_df["review_issue"] != ""
]

print("Suspicious translations found:", len(translation_review_df))

display(translation_review_df[
    ["sample", "segment_id", "text", "english_translation", "review_issue"]
].to_string(index=False))

Suspicious translations found: 1


'  sample  segment_id                                                                                                              text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        english_translation                          review_issue\nsample-7          80 जी हाँ, साल भर तो पढ़ाई करते हैं नहीं, इम्तिहान के समय बगले झांकते हैं और फिर आ जाते हैं यहां तक अपने नंबर बढ़वाने। Yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, 

In [50]:
row_index = translation_df[
    (translation_df["sample"] == "sample-7") &
    (translation_df["segment_id"] == 80)
].index[0]

hindi_text = translation_df.loc[row_index, "text"]

inputs = tokenizer(
    hindi_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

with torch.no_grad():
    outputs = translation_model.generate(
        **inputs,
        forced_bos_token_id=TARGET_TOKEN_ID,
        max_new_tokens=100,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )

new_translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("HINDI:", hindi_text)
print("RETRIED ENGLISH:", new_translation)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HINDI: जी हाँ, साल भर तो पढ़ाई करते हैं नहीं, इम्तिहान के समय बगले झांकते हैं और फिर आ जाते हैं यहां तक अपने नंबर बढ़वाने।
RETRIED ENGLISH: Yes, they study all year round, no, they stare at the exams and then come back to even increase their numbers.


In [51]:
# Preserve the original machine translation
translation_df["original_translation"] = translation_df["english_translation"]

# Replace only the problematic translation with the improved retry
translation_df.loc[row_index, "english_translation"] = new_translation

# Save the updated translations
translation_df.to_csv(
    OUTPUT_DIR / "sentence_translations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Updated translation saved successfully.")
print("Original:", translation_df.loc[row_index, "original_translation"])
print("Improved:", translation_df.loc[row_index, "english_translation"])

Updated translation saved successfully.
Original: Yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes, yes,
Improved: Yes, they study all year round, no, they stare at the exams and then come back to even increase their numbers.


In [52]:
print("Candidate groups columns:")
print(candidate_groups_df.columns.tolist())

print("\nFinal bilingual groups columns:")
print(final_groups_df.columns.tolist())

Candidate groups columns:
['sample', 'group_id', 'sentence_ids', 'group_size', 'hindi_text']

Final bilingual groups columns:
['sample', 'group_id', 'sentence_ids', 'group_size', 'hindi_text', 'english_translation']


In [53]:
display(
    final_groups_df[
        (final_groups_df["sample"] == "sample-7") &
        (final_groups_df["sentence_ids"].astype(str).str.contains(r"\b80\b"))
    ][["sample", "group_id", "sentence_ids", "hindi_text", "english_translation"]]
)

,sample,group_id,sentence_ids,hindi_text,english_translation
264,sample-7,sample-7_G48,[80],"जी हाँ, साल भर तो पढ़ाई करते हैं नहीं, इम्तिहान...","Yes, yes, yes, yes, yes, yes, yes, yes, yes, y..."


In [54]:
# Get the improved translation from sentence_translations
improved_translation = translation_df.loc[row_index, "english_translation"]

# Update only sample-7_G48
mask = (
    (final_groups_df["sample"] == "sample-7") &
    (final_groups_df["group_id"] == "sample-7_G48")
)

final_groups_df.loc[mask, "english_translation"] = improved_translation

# Save updated bilingual groups
final_groups_df.to_csv(
    OUTPUT_DIR / "final_bilingual_groups.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Updated group:")
print(final_groups_df.loc[mask, "english_translation"].iloc[0])
print("\nSaved final_bilingual_groups.csv")

Updated group:
Yes, they study all year round, no, they stare at the exams and then come back to even increase their numbers.

Saved final_bilingual_groups.csv


In [55]:
import shutil
from pathlib import Path

# Rebuild ZIP from the complete outputs folder
zip_path = shutil.make_archive(
    "/content/Hindi_Text_Processing_Outputs",
    "zip",
    root_dir=OUTPUT_DIR
)

print("Final ZIP created:", zip_path)
print("ZIP size:", round(Path(zip_path).stat().st_size / (1024 * 1024), 2), "MB")

from google.colab import files
files.download(zip_path)

Final ZIP created: /content/Hindi_Text_Processing_Outputs.zip
ZIP size: 0.67 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>